# Demo 02: Serial and Parallel Workflows Using CrewAI Agents

## Objective

Build two CrewAI workflows in Google Colab using the standard OpenAI API:

1. **Serial workflow:** The researcher completes a task first, and the writer uses its output.
2. **Parallel workflow:** Two independent tasks execute concurrently.

This notebook uses:

- Google Colab
- CrewAI
- OpenAI API
- Tavily web search
- Asynchronous CrewAI execution for Colab compatibility


## Scenario

A product leader needs timely insights about how AI is changing healthcare, drug discovery, and patient diagnostics.

The workflow uses:

- **AI Researcher:** Searches the web for current information.
- **Technical Writer:** Produces clear executive-ready reports.

The serial example demonstrates dependent tasks. The parallel example demonstrates independent tasks that can run concurrently.


In [1]:
# Step 2: Import libraries

import os
import getpass
import requests

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool


In [2]:
# Step 3: Securely enter API keys
# Values are hidden while typing and are not hard-coded in the notebook.

os.environ["ALOK_OPEN_API_KEY"] = os.getenv("ALOK_OPEN_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

if not os.environ["ALOK_OPEN_API_KEY"].strip():
    raise ValueError("OpenAI API key cannot be empty.")

if not os.environ["TAVILY_API_KEY"].strip():
    raise ValueError("Tavily API key cannot be empty.")

print("API keys configured successfully.")


API keys configured successfully.


In [3]:
# Step 4: Create the Tavily web-search tool

class TavilySearchTool(BaseTool):
    name: str = "Tavily Web Search"
    description: str = (
        "Search the web for recent and reliable information. "
        "Input should be a focused search query."
    )

    def _run(self, query: str) -> str:
        if not query or not query.strip():
            return "Search query cannot be empty."

        payload = {
            "api_key": os.environ["TAVILY_API_KEY"],
            "query": query,
            "search_depth": "advanced",
            "max_results": 5,
            "include_answer": False,
        }

        try:
            response = requests.post(
                "https://api.tavily.com/search",
                json=payload,
                timeout=30,
            )
            response.raise_for_status()
            data = response.json()
        except requests.RequestException as exc:
            return f"Web search failed: {exc}"
        except ValueError:
            return "Web search failed because Tavily returned an invalid response."

        results = data.get("results", [])
        if not results:
            return "No relevant web-search results were found."

        formatted_results = []
        for index, item in enumerate(results, start=1):
            title = item.get("title", "Untitled result")
            url = item.get("url", "URL unavailable")
            content = item.get("content", "No summary available")
            formatted_results.append(
                f"{index}. {title}\n"
                f"URL: {url}\n"
                f"Summary: {content}"
            )

        return "\n\n".join(formatted_results)


search_tool = TavilySearchTool()
print("Tavily search tool created.")


Tavily search tool created.


In [4]:
# Step 5: Configure the standard OpenAI model
# Replace the model name when another model is available in your OpenAI project.

llm = LLM(
    model="openai/gpt-4.1-nano",
    api_key=os.environ["ALOK_OPEN_API_KEY"],
    temperature=0.3,
)

print("OpenAI LLM configured successfully.")


OpenAI LLM configured successfully.


In [6]:
# Step 6: Define the agents

researcher = Agent(
    role="AI Healthcare Researcher",
    goal=(
        "Find recent and reliable advancements in healthcare AI, "
        "drug discovery, and medical technology."
    ),
    backstory=(
        "You are an AI researcher specializing in healthcare, drug discovery, "
        "clinical innovation, and emerging medical technologies. You verify "
        "important claims through web search and provide source URLs."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[search_tool],
)

writer = Agent(
    role="Technical Writer",
    goal="Convert technical AI research into concise executive reports.",
    backstory=(
        "You are an experienced technical writer who explains complex healthcare "
        "AI topics clearly for business and technology leaders."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

print("Agents created successfully.")


Agents created successfully.


# Part A: Serial Workflow

In a serial workflow, the second task depends on the first task.

```text
Research task → Research output → Writing task → Executive summary
```


In [7]:
# Step 7: Define serial tasks

task_research = Task(
    description=(
        "Use web search to identify the top three recent advancements in "
        "healthcare AI. For each advancement, explain the technology, its "
        "healthcare use, expected impact, one practical example, and a source URL. "
        "Use credible and recent sources."
    ),
    expected_output=(
        "Structured research notes describing exactly three healthcare AI "
        "advancements, including examples, business or clinical impact, and URLs."
    ),
    agent=researcher,
)

task_write = Task(
    description="""
Using the AI Researcher's notes, prepare an executive summary.

Requirements:
- Maximum 200 words
- Cover exactly three advancements
- Explain the value of each advancement
- Use clear headings or bullet points
- Preserve relevant source URLs
- Do not invent facts or sources
""",
    expected_output=(
        "An executive-ready summary of exactly three healthcare AI advancements, "
        "within 200 words."
    ),
    agent=writer,
    context=[task_research],
)

print("Serial tasks created.")


Serial tasks created.


In [8]:
# Step 8: Run the serial workflow
# Colab already runs an event loop, so CrewAI must be awaited asynchronously.

crew_serial = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    process=Process.sequential,
    verbose=True,
)

serial_result = await crew_serial.kickoff_async()

print("\n" + "=" * 70)
print("SERIAL WORKFLOW RESULT")
print("=" * 70)
print(serial_result.raw if hasattr(serial_result, "raw") else serial_result)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 8232e75d-f409-4ec5-abf1-149a49f5d3ed                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use web search to identify the top three recent advancements in healthcare AI. For each advancement,     │
│  explain the technology, its healthcare use, expected impact, one practical example, and a source URL. Use      │
│  credible and recent sources.                                                                                   │
│  ID: 37775ad8-9dcd-4840-bd88-b39dfdb5514d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Healthcare Researcher                                                                                │
│                                                                                                                 │
│  Task: Use web search to identify the top three recent advancements in healthcare AI. For each advancement,     │
│  explain the technology, its healthcare use, expected impact, one practical example, and a source URL. Use      │
│  credible and recent sources.                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: tavily_web_search                                                                                        │
│  Args: {'query': 'top recent advancements in healthcare AI 2023'}                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: tavily_web_search                                                                                        │
│  Args: {'query': 'latest breakthroughs in AI for medicine 2023'}                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Executive Summary of Healthcare AI Advancements in 2023**                                                    │
│                                                                                                                 │
│  - **Generative AI Enhances Clinical Decision-Making:**                                                         │
│  Generative AI models like Google’s Med-PaLM 2 and ChatGPT have achieved up to 86.5% accuracy on USMLE-style    │
│  questions, demonstrating their ability to understand complex medical queries. These models support             │
│  diagnostics, patient communication, and medical education, potentially improving diagnostic accuracy,          │
│  reducing physician workload, and enabling personalized medicine.                                               │
│  [Source](https://hmacademy.com/insights/AI-Catalyst/artificial-intelligence/five-healthcare-ai-trends-we-saw-  │
│  in-2023)                                                                                                       │
│                                                                                                                 │
│  - **AI-Driven Drug Discovery Accelerates Therapeutic Development:**                                            │
│  Platforms such as Insilico Medicine’s Pharma.AI and PandaOmics 4.0 incorporate chat functionalities,           │
│  off-target screening, and knowledge graphs to streamline target identification and compound design. This       │
│  accelerates drug development timelines, reduces costs, and facilitates personalized treatments, enhancing      │
│  responses to health crises like pandemics. [Source](https://www.eurekalert.org/news-releases/1009492)          │
│                                                                                                                 │
│  - **Advances in Medical Imaging and Remote Monitoring:**                                                       │
│  AI tools like DALL-E 3 improve imaging diagnostics, while wearable devices monitor conditions such as          │
│  arrhythmias in real-time. These innovations enable faster, higher-resolution diagnostics, early detection of   │
│  health issues, and expanded access to care outside clinical settings, leading to better management of chronic  │
│  diseases and proactive interventions.                                                                          │
│  [Source](https://www.philips.com/a-w/about/news/archive/features/2023/20230116-10-healthcare-technology-trend  │
│  s-for-2023.html)                                                                                               │
│                                                                                                                 │
│  These advancements collectively promise more accurate, efficient, and personalized healthcare delivery.        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Using the AI Researcher's notes, prepare an executive summary.                                                 │
│                                                                                                                 │
│  Requirements:                                                                                                  │
│  - Maximum 200 words                                                                                            │
│  - Cover exactly three advancements                                                                             │
│  - Explain the value of each advancement                                                                        │
│  - Use clear headings or bullet points                                                                          │
│  - Preserve relevant source URLs                                                                                │
│  - Do not invent facts or sources                                                                               │
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


SERIAL WORKFLOW RESULT
**Executive Summary of Healthcare AI Advancements in 2023**

- **Generative AI Enhances Clinical Decision-Making:**  
Generative AI models like Google’s Med-PaLM 2 and ChatGPT have achieved up to 86.5% accuracy on USMLE-style questions, demonstrating their ability to understand complex medical queries. These models support diagnostics, patient communication, and medical education, potentially improving diagnostic accuracy, reducing physician workload, and enabling personalized medicine. [Source](https://hmacademy.com/insights/AI-Catalyst/artificial-intelligence/five-healthcare-ai-trends-we-saw-in-2023)

- **AI-Driven Drug Discovery Accelerates Therapeutic Development:**  
Platforms such as Insilico Medicine’s Pharma.AI and PandaOmics 4.0 incorporate chat functionalities, off-target screening, and knowledge graphs to streamline target identification and compound design. This accelerates drug development timelines, reduces costs, and facilitates personalized treat

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 8232e75d-f409-4ec5-abf1-149a49f5d3ed                                                                       │
│  Final Output: **Executive Summary of Healthcare AI Advancements in 2023**                                      │
│                                                                                                                 │
│  - **Generative AI Enhances Clinical Decision-Making:**                                                         │
│  Generative AI models like Google’s Med-PaLM 2 and ChatGPT have achieved up to 86.5% accuracy on USMLE-style    │
│  questions, demonstrating their ability to understand complex medical queries. These models support             │
│  diagnostics, patient communication, and medical education, potentially improving diagnostic accuracy,          │
│  reducing physician workload, and enabling personalized medicine.                                               │
│  [Source](https://hmacademy.com/insights/AI-Catalyst/artificial-intelligence/five-healthcare-ai-trends-we-saw-  │
│  in-2023)                                                                                                       │
│                                                                                                                 │
│  - **AI-Driven Drug Discovery Accelerates Therapeutic Development:**                                            │
│  Platforms such as Insilico Medicine’s Pharma.AI and PandaOmics 4.0 incorporate chat functionalities,           │
│  off-target screening, and knowledge graphs to streamline target identification and compound design. This       │
│  accelerates drug development timelines, reduces costs, and facilitates personalized treatments, enhancing      │
│  responses to health crises like pandemics. [Source](https://www.eurekalert.org/news-releases/1009492)          │
│                                                                                                                 │
│  - **Advances in Medical Imaging and Remote Monitoring:**                                                       │
│  AI tools like DALL-E 3 improve imaging diagnostics, while wearable devices monitor conditions such as          │
│  arrhythmias in real-time. These innovations enable faster, higher-resolution diagnostics, early detection of   │
│  health issues, and expanded access to care outside clinical settings, leading to better management of chronic  │
│  diseases and proactive interventions.                                                                          │
│  [Source](https://www.philips.com/a-w/about/news/archive/features/2023/20230116-10-healthcare-technology-trend  │
│  s-for-2023.html)                                                                                               │
│                                                                                                                 │
│  These advancements collectively promise more accurate, efficient, and personalized healthcare delivery.        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# Part B: Parallel Workflow

Parallel tasks must be independent. Both tasks below are marked with `async_execution=True`.

```text
Drug-discovery research ─┐
                         ├─ Run concurrently
Diagnostics report ──────┘
```

A final consolidation task is added because CrewAI expects the last task to be synchronous.


In [9]:
# Step 9: Define parallel tasks

task_parallel_1 = Task(
    description=(
        "Use web search to identify five companies applying AI to drug discovery. "
        "For each company, provide its name, specialization, one example technology "
        "or program, and a source URL."
    ),
    expected_output=(
        "A structured list of five AI drug-discovery companies with short "
        "descriptions and source URLs."
    ),
    async_execution=True,
    agent=researcher,
)

task_parallel_2 = Task(
    description=(
        "Write a concise report explaining how AI is transforming patient "
        "diagnostics. Cover medical imaging, predictive analytics, multimodal AI, "
        "and clinical decision support. Limit the report to about 200 words."
    ),
    expected_output=(
        "A concise report on AI-driven patient diagnostics with practical examples."
    ),
    async_execution=True,
    agent=writer,
)

task_parallel_summary = Task(
    description="""
Combine the outputs of the two parallel tasks into one final report.

Create two sections:
1. AI Companies in Drug Discovery
2. AI Transformation in Patient Diagnostics

Preserve important source URLs from the research output.
Do not add unsupported facts.
""",
    expected_output=(
        "A consolidated report containing the drug-discovery company findings "
        "and the patient-diagnostics report."
    ),
    agent=writer,
    context=[task_parallel_1, task_parallel_2],
)

print("Parallel tasks created.")


Parallel tasks created.


In [10]:
# Step 10: Run the parallel workflow asynchronously

crew_parallel = Crew(
    agents=[researcher, writer],
    tasks=[task_parallel_1, task_parallel_2, task_parallel_summary],
    process=Process.sequential,
    verbose=True,
)

parallel_result = await crew_parallel.kickoff_async()

print("\n" + "=" * 70)
print("PARALLEL WORKFLOW RESULT")
print("=" * 70)
print(parallel_result.raw if hasattr(parallel_result, "raw") else parallel_result)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a00e2cf8-0124-4135-b9be-1c3b644aeb53                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Write a concise report explaining how AI is transforming patient diagnostics. Cover medical imaging,     │
│  predictive analytics, multimodal AI, and clinical decision support. Limit the report to about 200 words.       │
│  ID: 5d347f8b-7afe-490d-8c52-628a6d01c667                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use web search to identify five companies applying AI to drug discovery. For each company, provide its   │
│  name, specialization, one example technology or program, and a source URL.                                     │
│  ID: 72e1a2d6-d03a-4f41-98e1-a7cfefe3b343                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Healthcare Researcher                                                                                │
│                                                                                                                 │
│  Task: Use web search to identify five companies applying AI to drug discovery. For each company, provide its   │
│  name, specialization, one example technology or program, and a source URL.                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Task: Write a concise report explaining how AI is transforming patient diagnostics. Cover medical imaging,     │
│  predictive analytics, multimodal AI, and clinical decision support. Limit the report to about 200 words.       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Consolidated Report on AI in Healthcare                                                                      │
│                                                                                                                 │
│  ## 1. AI Companies in Drug Discovery                                                                           │
│                                                                                                                 │
│  The integration of artificial intelligence (AI) in drug discovery is reshaping the pharmaceutical landscape,   │
│  enabling faster and more efficient development of new therapies. Below are five notable companies leading      │
│  this transformation:                                                                                           │
│                                                                                                                 │
│  1. **Recursion Pharmaceuticals**                                                                               │
│     - **Specialization**: AI-driven drug discovery focusing on rare diseases and oncology.                      │
│     - **Example Technology/Program**: Collaboration with NVIDIA to enhance AI models for drug discovery using   │
│  a 23-petabyte dataset.                                                                                         │
│     - **Source URL**: [Recursion lines up $50M Nvidia collab for AI drug                                        │
│  discovery](https://www.fiercebiotech.com/medtech/recursion-lines-50m-nvidia-collab-ai-powered-drug-discovery)  │
│                                                                                                                 │
│  2. **Insilico Medicine**                                                                                       │
│     - **Specialization**: Generative AI for drug discovery and development, particularly in oncology and        │
│  aging.                                                                                                         │
│     - **Example Technology/Program**: Pharma.AI platform, which includes the first generative AI drug,          │
│  INS018_055, entering Phase II trials.                                                                          │
│     - **Source URL**: [First Generative AI Drug Begins Phase II Trials with                                     │
│  Patients](https://insilico.com/blog/first_phase2)                                                              │
│                                                                                                                 │
│  3. **Exscientia**                                                                                              │
│     - **Specialization**: AI-designed drug candidates, particularly in oncology.                                │
│     - **Example Technology/Program**: Centaur Chemist platform, which accelerates drug design and development   │
│  processes.                                                                                                     │
│     - **Source URL**: [Exscientia Uses Generative AI to Reimagine Drug                                          │
│  Discovery](https://aws.amazon.com/solutions/case-studies/exscientia-generative-ai)                             │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Combine the outputs of the two parallel tasks into one final report.                                           │
│                                                                                                                 │
│  Create two sections:                                                                                           │
│  1. AI Companies in Drug Discovery                                                                              │
│  2. AI Transformation in Patient Diagnostics                                                                    │
│                                                                                                                 │
│  Preserve important source URLs from the research output.                                                       │
│  Do not add unsupported facts.                                                                                  │
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


PARALLEL WORKFLOW RESULT
# Consolidated Report on AI in Healthcare

## 1. AI Companies in Drug Discovery

The integration of artificial intelligence (AI) in drug discovery is reshaping the pharmaceutical landscape, enabling faster and more efficient development of new therapies. Below are five notable companies leading this transformation:

1. **Recursion Pharmaceuticals**
   - **Specialization**: AI-driven drug discovery focusing on rare diseases and oncology.
   - **Example Technology/Program**: Collaboration with NVIDIA to enhance AI models for drug discovery using a 23-petabyte dataset.
   - **Source URL**: [Recursion lines up $50M Nvidia collab for AI drug discovery](https://www.fiercebiotech.com/medtech/recursion-lines-50m-nvidia-collab-ai-powered-drug-discovery)

2. **Insilico Medicine**
   - **Specialization**: Generative AI for drug discovery and development, particularly in oncology and aging.
   - **Example Technology/Program**: Pharma.AI platform, which includes the first 

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a00e2cf8-0124-4135-b9be-1c3b644aeb53                                                                       │
│  Final Output: # Consolidated Report on AI in Healthcare                                                        │
│                                                                                                                 │
│  ## 1. AI Companies in Drug Discovery                                                                           │
│                                                                                                                 │
│  The integration of artificial intelligence (AI) in drug discovery is reshaping the pharmaceutical landscape,   │
│  enabling faster and more efficient development of new therapies. Below are five notable companies leading      │
│  this transformation:                                                                                           │
│                                                                                                                 │
│  1. **Recursion Pharmaceuticals**                                                                               │
│     - **Specialization**: AI-driven drug discovery focusing on rare diseases and oncology.                      │
│     - **Example Technology/Program**: Collaboration with NVIDIA to enhance AI models for drug discovery using   │
│  a 23-petabyte dataset.                                                                                         │
│     - **Source URL**: [Recursion lines up $50M Nvidia collab for AI drug                                        │
│  discovery](https://www.fiercebiotech.com/medtech/recursion-lines-50m-nvidia-collab-ai-powered-drug-discovery)  │
│                                                                                                                 │
│  2. **Insilico Medicine**                                                                                       │
│     - **Specialization**: Generative AI for drug discovery and development, particularly in oncology and        │
│  aging.                                                                                                         │
│     - **Example Technology/Program**: Pharma.AI platform, which includes the first generative AI drug,          │
│  INS018_055, entering Phase II trials.                                                                          │
│     - **Source URL**: [First Generative AI Drug Begins Phase II Trials with                                     │
│  Patients](https://insilico.com/blog/first_phase2)                                                              │
│                                                                                                                 │
│  3. **Exscientia**                                                                                              │
│     - **Specialization**: AI-designed drug candidates, particularly in oncology.                                │
│     - **Example Technology/Program**: Centaur Chemist platform, which accelerates drug design and development   │
│  processes.                                                                                                     │
│     - **Source URL**: [Exscientia Uses Generative AI to Reimagine Drug                                          │
│  Discovery](https://aws.amazon.com/solutions/case-studies/exscientia-generative-ai)                             │
│                                                       

In [12]:
output_file = "/content/fmcg_ai_executive_summary.md"

with open(output_file, "w", encoding="utf-8") as file:
    file.write(parallel_result.raw)

print(f"Report saved successfully: {output_file}")


Report saved successfully: /content/fmcg_ai_executive_summary.md


## Key Concepts

### Serial execution

Use serial execution when tasks depend on earlier outputs. In this notebook, the writer cannot prepare the executive summary until the researcher completes the healthcare AI research.

### Parallel execution

Use parallel execution when tasks are independent. The drug-discovery research and diagnostics report can run at the same time.

### Why `await crew.kickoff_async()` is used

Google Colab and Jupyter already run an asynchronous event loop. Calling `crew.kickoff()` may produce:

```text
RuntimeError: Agent execution was invoked synchronously from within a running event loop
```

Therefore, both workflows use:

```python
result = await crew.kickoff_async()
```

### Security

- Never hard-code OpenAI or Tavily API keys.
- Revoke any API key that has previously been exposed.
- Colab environment variables are cleared when the runtime is reset.
